DROWSINESS DETECTION - WEBCAM 

In [1]:
# IMPORTING THE REQUIRED LIBRARIES
import cv2
import time
import numpy as np
import mediapipe as mp
import tensorflow as tf
import os
import urllib.request

C:\Users\Sanjib Das\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Sanjib Das\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Sanjib Das\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle

In [2]:
# CONFIGURATION
IMG_SIZE = 64
CLASSES = ["Alert", "Drowsy_Eyes_Closed", "Drowsy_Yawning"]
MODEL_PATH = "Drowsiness_model"          
WEIGHTS_PATH = "drowsiness_model_weights.weights.h5"  
LANDMARKER_PATH = "face_landmarker.task"

# AUTO-DOWNLOAD face_landmarker.task IF MISSING
if not os.path.exists(LANDMARKER_PATH):
    print("Downloading face_landmarker.task...")
    url = (
        "https://storage.googleapis.com/mediapipe-models/"
        "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    )
    urllib.request.urlretrieve(url, LANDMARKER_PATH)
    print("Downloaded face_landmarker.task")

# LOAD MODEL
print("Loading model...")

def build_model():
    """Rebuild the exact same architecture used in training."""
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
    model = Sequential([
        Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        Conv2D(16, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Conv2D(32, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'), MaxPooling2D(2, 2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

if os.path.exists(MODEL_PATH):
    try:
        model = tf.saved_model.load(MODEL_PATH)
        # Wrap for easy prediction
        infer = model.signatures["serving_default"]
        USE_SAVED_MODEL = True
        print(f"Loaded SavedModel from '{MODEL_PATH}'")
    except Exception as e:
        print(f"SavedModel load failed: {e}")
        USE_SAVED_MODEL = False
elif os.path.exists(WEIGHTS_PATH):
    # Fallback: rebuild architecture and load weights
    model = build_model()
    model.load_weights(WEIGHTS_PATH)
    USE_SAVED_MODEL = False
    print(f"Loaded weights from '{WEIGHTS_PATH}'")
else:
    #Try loading keras directly
    try:
        model = tf.keras.models.load_model(
            'drowsiness_detection_model.keras',
            compile=False  # skip compile to avoid config issues
        )
        USE_SAVED_MODEL = False
        print("Loaded .keras model with compile=False")
    except Exception as e:
        print(f"ERROR: Could not load model. {e}")
        print("Please run colab_export_model.py on Colab first.")
        exit(1)

def predict(frame_input):
    """Run inference regardless of model type."""
    if USE_SAVED_MODEL:
        tensor = tf.constant(frame_input, dtype=tf.float32)
        # Auto-detect the input key name
        input_key = list(infer.structured_input_signature[1].keys())[0]
        result = infer(**{input_key: tensor})
        # Auto-detect the output key name 
        output_key = list(result.keys())[0]
        return result[output_key].numpy()[0]
    else:
        return model.predict(frame_input, verbose=0)[0]

print("Model ready.")

# MEDIAPIPE FACE LANDMARKER SETUP 
print("Initialising MediaPipe Face Landmarker...")

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=LANDMARKER_PATH),
    running_mode=RunningMode.VIDEO,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

detector = FaceLandmarker.create_from_options(options)
print("MediaPipe ready.")

# COLOUR MAP FOR LABELS
LABEL_COLORS = {
    "Alert":              (0, 220, 100),   # green
    "Drowsy_Eyes_Closed": (0, 100, 255),   # orange-red
    "Drowsy_Yawning":     (0, 200, 255),   # yellow
}

print("Opening webcam... Press Q to quit.")
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("ERROR: Could not open webcam.")
    exit(1)

# Smoothing
from collections import deque, Counter
pred_history = deque(maxlen=15) 

LABEL_LOCK_SECONDS = 1.0
locked_label    = None
locked_color    = (180, 180, 180)
lock_until_time = 0.0

# Confidence threshold 
CONFIDENCE_THRESHOLD = 0.75
frame_ts = 0  

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]
    frame_ts += 33  

    # MediaPipe detection
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect_for_video(mp_image, frame_ts)

    label = "No Face"
    color = (180, 180, 180)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]

        xs = [int(lm.x * w) for lm in landmarks]
        ys = [int(lm.y * h) for lm in landmarks]

        for x, y in zip(xs, ys):
            cv2.circle(frame, (x, y), 1, (0, 255, 100), -1)

        x_min = max(min(xs) - 20, 0)
        y_min = max(min(ys) - 30, 0)
        x_max = min(max(xs) + 20, w)
        y_max = min(max(ys) + 20, h)

        face_crop = frame[y_min:y_max, x_min:x_max]

        if face_crop.size != 0:
            gray      = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
            resized   = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
            normalized = resized / 255.0
            reshaped  = normalized.reshape(1, IMG_SIZE, IMG_SIZE, 1).astype(np.float32)

            # Predict
            probs = predict(reshaped)
            class_idx = int(np.argmax(probs))
            label = CLASSES[class_idx]
            confidence = probs[class_idx]

            pred_history.append(class_idx)
            smooth_idx   = Counter(pred_history).most_common(1)[0][0]
            smooth_label = CLASSES[smooth_idx]

            now = time.time()
            if confidence < CONFIDENCE_THRESHOLD:
                smooth_label = "Uncertain"
                color        = (180, 180, 180)
            else:
                color = LABEL_COLORS[smooth_label]

            if now >= lock_until_time:
                locked_label    = smooth_label
                locked_color    = color
                lock_until_time = now + LABEL_LOCK_SECONDS

            display_label = locked_label if locked_label else smooth_label
            display_color = locked_color

            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), display_color, 2)

            main_text = f"{display_label}  {confidence:.0%}"
            cv2.putText(frame, main_text, (x_min, y_min - 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, display_color, 2)

            for i, cls in enumerate(CLASSES):
                bar_label = f"{cls}: {probs[i]:.0%}"
                bar_color = LABEL_COLORS[cls]
                cv2.putText(frame, bar_label, (10, 30 + i * 28),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, bar_color, 2)

    else:
        cv2.putText(frame, "No face detected", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (180, 180, 180), 2)

    # FPS counter
    cv2.putText(frame, "Press Q to quit", (w - 160, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    cv2.imshow("Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
detector.close()
cv2.destroyAllWindows()
print("Done.")

Loading model...
Loaded SavedModel from 'Drowsiness_model'
Model ready.
Initialising MediaPipe Face Landmarker...
MediaPipe ready.
Opening webcam... Press Q to quit.
Done.
